# ie-agent walkthrough, variant: examples with every section

Same agent as in `walkthrough.ipynb`, with one change (`examples_with_sections: true` in `config.yaml`):

| | default | this variant |
|---|---|---|
| title | 5 title examples in the first message | the same |
| `read_abstract_intro` | returns the abstract or introduction | returns it **plus 5 examples found via the abstract** |
| `read_toc` | returns the table of contents | returns it **plus 5 examples found via the table of contents** |
| `get_examples` | optional tool, the model decides | not offered: section and examples always come together |

Why: in the pipeline conditions A–F the model always gets the examples of the section it reads. In the default
agent Qwen asked for extra examples in only 30 % of the documents. With this variant agent and pipeline see the
same examples, which makes the comparison fairer.

**Needs:** Weaviate (8087), TEI (8090), the GND data (`uv run ie-agent build-gnd`); Part 3 a vLLM server.

In [1]:
import json
import textwrap
from types import SimpleNamespace

import yaml

from ie_agent.agent import Agent
from ie_agent.config import CONFIG
from ie_agent.data import Documents, Gnd, Mapper, load_gold
from ie_agent.tools import check_call, new_state, offered_tools, stage

DOC_ID = "1007748524"   # "Optische Spektroskopie an zweidimensionalen Lochsystemen ...", no duplicate in training


def show(text, width=110, limit=1500):
    """Prints long text wrapped and shortened."""
    text = str(text)
    print(textwrap.fill(text[:limit], width, replace_whitespace=False) + (" …" if len(text) > limit else ""))


def names(tools):
    return [t["function"]["name"] for t in tools]


cfg = yaml.safe_load(CONFIG.read_text())
cfg["agent"]["examples_with_sections"] = True          # <- the variant
docs, gnd, mapper, gold = Documents(cfg), Gnd.load(cfg), Mapper(cfg), load_gold(cfg)

print("TITLE:", docs.title[DOC_ID])
print("GOLD: ", sorted(gnd.label(i) for i in gold[DOC_ID]))

TITLE: Optische Spektroskopie an zweidimensionalen Lochsystemen in GaAs-AlGaAs-Quantenwells und an GaAs-AlGaAs-GaMnAs-Schichtstrukturen
GOLD:  ['Galliumarsenid', 'Heterostruktur', 'Optische Spektroskopie']


## Part 1: how much do the examples of each section help?

Before the agent runs: how many gold headings already appear in the five examples of each section? These are the
examples the model gets in this variant.

In [2]:
for section in ["title", "abstract_intro", "toc"]:
    examples = docs.examples[section].get(DOC_ID, [])
    headings = {x.strip() for _, _, ids in examples for x in str(ids).split(",")}
    in_gold = sorted(gnd.label(i) for i in headings & gold[DOC_ID])
    print(f"{section:15s} {len(examples)} examples, {len(headings):2d} different headings, of them in gold: {in_gold}")

title           5 examples, 35 different headings, of them in gold: ['Optische Spektroskopie']
abstract_intro  5 examples, 25 different headings, of them in gold: ['Galliumarsenid', 'Heterostruktur']
toc             5 examples, 21 different headings, of them in gold: ['Galliumarsenid', 'Heterostruktur']


## Part 2: one run by hand

### 2.1 The prompt

Only point 4 of the system prompt differs from the default: the model is told that every section comes with
examples, and `get_examples()` is not mentioned.

In [3]:
agent = Agent({**cfg, "llm": {**cfg["llm"], "model": "(none)"}}, docs, mapper, gnd, client=SimpleNamespace())
print([line for line in agent.system_prompt.splitlines() if line.startswith("4.")][0])
print()
print("=== FIRST MESSAGE ===")
print(agent.first_message.format(title=docs.title[DOC_ID], examples=agent.tools.examples(DOC_ID, "title")))

4. Die Beispiele zum Titel hast du schon. Mit jedem gelesenen Abschnitt bekommst du automatisch 5 ähnliche, bereits erschlossene Dokumente zu diesem Abschnitt.

=== FIRST MESSAGE ===
Titel: Optische Spektroskopie an zweidimensionalen Lochsystemen in GaAs-AlGaAs-Quantenwells und an GaAs-AlGaAs-GaMnAs-Schichtstrukturen

Ähnliche erschlossene Titel:

Beispiel 1: Mikroskopie und optische Spektroskopie an heterogenen Nano- und Mikrostrukturen: Halbleiter-Nanokristalle, molekulare Farbstofffilme und funktionalisierte Hybridstrukturen
Schlagwörter: Mikroskopie; Optische Spektroskopie; Mikrostruktur; Nanostruktur

Beispiel 2: Optische Spektroskopie an Chalkogenen und Porphyrinen. Optical Spectroscopy on chalcogens and porphyrins
Schlagwörter: Verbindungshalbleiter; Chalkogenide; Photolumineszenzspektroskopie; Porphyrine; Photostrom

Beispiel 3: Optische Verformungsmessung an heterogenen Werkstoffsystemen
Schlagwörter: Polypropylen; Zugversuch; Optische Messung; Rasterelektronenmikroskop; Faser

### 2.2 Offered tools per stage: default and variant side by side

The read tools get one more sentence in their description, and `get_examples` disappears.

In [4]:
def call(name, **args):
    """What happens when the model calls a tool: policy check first, then the tool."""
    refused = check_call(name, args, state)
    print(f"--> {name}({json.dumps(args, ensure_ascii=False)})", "REFUSED" if refused else "")
    return refused or agent.tools.run(DOC_ID, name, args, state)


def compare_offer():
    print(f"stage {stage(state)}")
    print("   default:", names(offered_tools(state, last_step=False)))
    print("   variant:", names(offered_tools(state, last_step=False, examples_with_sections=True)))


state = new_state(docs.title[DOC_ID])
compare_offer()
print()
print("description of read_abstract_intro in the variant:")
print("  ", offered_tools(state, False, examples_with_sections=True)[0]["function"]["description"])

stage title
   default: ['read_abstract_intro', 'map_to_gnd']
   variant: ['read_abstract_intro', 'map_to_gnd']

description of read_abstract_intro in the variant:
   Liefert das Abstract des Dokuments oder, falls es keines gibt, den Anfang der Einleitung. Dazu kommen 5 ähnliche, bereits erschlossene Dokumente, gesucht über diesen Abschnitt.


### 2.3 `read_abstract_intro`: section and examples in one answer

In [5]:
result = call("read_abstract_intro")
section, examples = result.split("\n\nÄhnliche erschlossene Dokumente zu diesem Abschnitt:\n\n")
show(section, limit=700)
print()
print("=== EXAMPLES THAT CAME WITH IT ===")
print(examples)
print()
compare_offer()

--> read_abstract_intro({}) 
[Einleitung]
Die immer weiter fortschreitende Miniaturisierung der Strukturgrößen und damit eine Erhöhung der
Komplexität integrierter Schaltkreise folgt seit den 70er Jahren dem bekannten Moore'schen Gesetz [Moo65].
Neben technologischen Herausforderungen in der Prozessierung von Halbleiterbauelementen werden mit immer
kleineren Strukturgrößen zusätzlich Quanteneffekte die Arbeitsweise zukünftiger Bauelemente dominieren. Ein
aktuelles Forschungsgebiet ist daher die Spintronik, die neben der Ladung des Elektrons auch dessen
magnetisches Moment, den Spin, zur Informationsspeicherung und -verarbeitung zugänglich machen will [Wol01].
Eine Anwendung aus der Spintronik ist beispielsweise der Ries …

=== EXAMPLES THAT CAME WITH IT ===
Beispiel 1: Seit der Vorstellung des Germanium-Transistors [Bar49] wurde der technische Fortschritt maßgeblich von den rasanten Entwicklungen in der Halbleiter-Mikroelektronik getragen. Bauelemente zur Informationsverarbeitung und -

### 2.4 `map_to_gnd`

Candidates from title, introduction and the examples. GaMnAs has no own GND term: it lands on "Galliumarsenid"
with a low score.

In [6]:
for hit in json.loads(call("map_to_gnd", candidates=["Optische Spektroskopie", "Quantenwell", "Galliumarsenid",
                                                     "Aluminiumgalliumarsenid", "Galliummanganarsenid",
                                                     "Spinrelaxation"])):
    print(hit)
print()
compare_offer()

--> map_to_gnd({"candidates": ["Optische Spektroskopie", "Quantenwell", "Galliumarsenid", "Aluminiumgalliumarsenid", "Galliummanganarsenid", "Spinrelaxation"]}) 


{'kandidat': 'Optische Spektroskopie', 'gnd_id': '040436802', 'gnd_begriff': 'Optische Spektroskopie', 'score': 1.0}
{'kandidat': 'Quantenwell', 'gnd_id': '041240103', 'gnd_begriff': 'Quantenwell', 'score': 1.0}
{'kandidat': 'Galliumarsenid', 'gnd_id': '040191559', 'gnd_begriff': 'Galliumarsenid', 'score': 1.0}
{'kandidat': 'Aluminiumgalliumarsenid', 'gnd_id': '1135930449', 'gnd_begriff': 'Aluminiumgalliumarsenid', 'score': 1.0}
{'kandidat': 'Galliummanganarsenid', 'gnd_id': '040191559', 'gnd_begriff': 'Galliumarsenid', 'score': 0.7}
{'kandidat': 'Spinrelaxation', 'gnd_id': '041823338', 'gnd_begriff': 'Spinrelaxation', 'score': 1.0}

stage mapped
   default: ['read_toc', 'get_examples', 'map_to_gnd', 'gnd_context']
   variant: ['read_toc', 'map_to_gnd', 'gnd_context']


### 2.5 `read_toc`: again section and examples together

In [7]:
result = call("read_toc")
section, examples = result.split("\n\nÄhnliche erschlossene Dokumente zu diesem Abschnitt:\n\n")
show(section, limit=600)
print()
print("=== EXAMPLES THAT CAME WITH IT ===")
print(examples)
print()
print("examples the model has seen so far:", state["examples"])

--> read_toc({}) 
Wachstum und Materialeigenschaften
Molekularstrahlepitaxie
Eigenschaften von Al x Ga 1-x As-Strukturen
Kristallstruktur
Bandstruktur und elektronische Zustände von Al x Ga 1-x As
Heterostrukturen und Dotierung
Neutrale und geladene Exzitonen
Wachstum und Eigenschaften von GaMnAs
Tieftemperatur GaAs
Kristalleigenschaften: LT-GaMnAs
Annealing
Ferromagnetische Kopplung in GaMnAs
Spin in Halbleitern
Spin-Bahn-
Wechselwirkung
Bulk und structure inversion asymmetry
Optische Erzeugung spinpolarisierter Ladungsträger
Spinrelaxation und Spindephasierung
Spinrelaxationsmechanismen
Der Elliott-Yafet-Mecha …

=== EXAMPLES THAT CAME WITH IT ===
Beispiel 1: Grundlagen der Gruppe III-Nitride Kristallstruktur und Polarisation Spontane Polarisation Piezoelektrische Polarisation, Verspannung und Relaxation Polarisationsinduzierte Ladungsträgerdichten Polarisation in Al Ga x 1 - x N-Schichten homogener Zusammensetzung Polarisation in GaN/Al Ga x 1 - x N-Heterostrukturen Polarisation in 

The table of contents is about growth and heterostructures, and the examples confirm it. So "Heterostruktur"
becomes a candidate, a gold heading that both models missed in the default variant.

In [8]:
for hit in json.loads(call("map_to_gnd", candidates=["Heterostruktur", "Molekularstrahlepitaxie"])):
    print(hit)

--> map_to_gnd({"candidates": ["Heterostruktur", "Molekularstrahlepitaxie"]}) 
{'kandidat': 'Heterostruktur', 'gnd_id': '041233786', 'gnd_begriff': 'Heterostruktur', 'score': 1.0}
{'kandidat': 'Molekularstrahlepitaxie', 'gnd_id': '041703995', 'gnd_begriff': 'Molekularstrahlepitaxie', 'score': 1.0}


### 2.6 `gnd_context` and `finish`

At most 8 IDs per call (`max_context_ids`), so we check the candidates with a perfect score.

In [9]:
ids = [i for i, m in state["mapped"].items() if m["score"] == 1.0][:8]
for entry in json.loads(call("gnd_context", gnd_ids=ids)):
    print(f"{entry['begriff']:28s} assigned {entry['in_trainingsdokumenten_vergeben']:>4}x | {entry['sachgruppen']} "
          f"| broader {[b['begriff'] for b in entry['oberbegriffe']]}")
    if entry["beziehungen_zu_deinen_kandidaten"]:
        print(" " * 30, entry["beziehungen_zu_deinen_kandidaten"])

--> gnd_context({"gnd_ids": ["040436802", "041240103", "040191559", "1135930449", "041823338", "041233786", "041703995"]}) 
Optische Spektroskopie       assigned    2x | ['Elektrizität, Magnetismus, Optik'] | broader ['Spektroskopie']
Quantenwell                  assigned    2x | ['Plasma, Gas, Flüssigkeit, Festkörper'] | broader ['Nanostruktur', 'Heterostruktur']
                               ['ist Unterbegriff deines Kandidaten „Heterostruktur“']
Galliumarsenid               assigned    6x | ['Anorganische Chemie'] | broader ['Drei-Fünf-Halbleiter']
AlGaAs                       assigned    0x | [] | broader ['Drei-Fünf-Halbleiter', 'Arsenide']
Spinrelaxation               assigned    0x | ['Elementarteilchen, Kern-, Atom-, Molekularphysik', 'Plasma, Gas, Flüssigkeit, Festkörper'] | broader ['Relaxation']
Heterostruktur               assigned    8x | ['Plasma, Gas, Flüssigkeit, Festkörper'] | broader ['Ungeordnetes System']
                               ['ist Oberbegriff deines Kand

`gnd_context` reports that Quantenwell is a narrower term of Heterostruktur. The RSWK rule in the prompt says
to drop the broader term then, but the indexers assigned Heterostruktur here. In this hand-made run we keep both, to
show the trade-off: Heterostruktur is a hit, Quantenwell costs precision.

In [10]:
def score(predicted, doc_id=DOC_ID):
    predicted, g = set(predicted), gold[doc_id]
    hits = len(predicted & g)
    p = hits / len(predicted) if predicted else 0.0
    r = hits / len(g)
    f1 = 2 * p * r / (p + r) if hits else 0.0
    print(f"P {p:.2f} | R {r:.2f} | F1 {f1:.2f}")
    print("hits:  ", sorted(gnd.label(i) for i in predicted & g))
    print("extra: ", sorted(gnd.label(i) for i in predicted - g))
    print("missed:", sorted(gnd.label(i) for i in g - predicted))


by_term = {m["term"]: i for i, m in state["mapped"].items()}
chosen = [by_term[t] for t in ["Optische Spektroskopie", "Galliumarsenid", "Heterostruktur", "Quantenwell"]
          if t in by_term]
score(chosen)

P 0.75 | R 1.00 | F1 0.86
hits:   ['Galliumarsenid', 'Heterostruktur', 'Optische Spektroskopie']
extra:  ['Quantenwell']
missed: []


## Part 3: the real model with this variant

Set `RUN_LIVE = True` and the port of a running vLLM server. The wrapper prints every step with the start of the
reasoning. Compare with Part 3 of `walkthrough.ipynb` (default variant, same document): Qwen read only the
introduction, never asked for examples, and missed "Heterostruktur" (P 0.33, R 0.67, F1 0.44).

In [11]:
RUN_LIVE = True
PORT = 9514          # vLLM server, e.g. Qwen on GPU 0
THINKING = True      # reasoning on (max_tokens is raised to 8192)

if RUN_LIVE:
    from openai import OpenAI

    class PrintingClient:
        """Passes every request to the server and prints what goes in and out."""

        def __init__(self, client):
            self.client = client
            self.models = client.models
            self.chat = SimpleNamespace(completions=SimpleNamespace(create=self.create))
            self.step = 0

        def create(self, **kwargs):
            self.step += 1
            last = kwargs["messages"][-1]
            if last["role"] == "tool":
                print(f"   tool result: {len(last['content'])} characters")
            print(f"step {self.step} | offered {names(kwargs['tools'])}"
                  + (" | FORCED finish" if kwargs["tool_choice"] != "auto" else ""))
            response = self.client.chat.completions.create(**kwargs)
            msg = response.choices[0].message
            reasoning = getattr(msg, "reasoning_content", None) or getattr(msg, "reasoning", None)
            if reasoning:
                print("   reasoning:", textwrap.shorten(reasoning, 400))
            for c in msg.tool_calls or []:
                print(f"   model calls {c.function.name}({c.function.arguments[:150]})")
            if not msg.tool_calls:
                print("   model answers with text:", (msg.content or "")[:200])
            return response

    live_cfg = {**cfg, "llm": {**cfg["llm"], "base_url": f"http://localhost:{PORT}/v1", "reasoning": THINKING, "model": None,
                               "max_tokens": 8192 if THINKING else 2048}}
    client = PrintingClient(OpenAI(base_url=live_cfg["llm"]["base_url"], api_key="unused"))
    live_agent = Agent(live_cfg, docs, mapper, gnd, client=client)
    print("model:", live_agent.model, "| prompt hash:", live_agent.prompt_hash)
    print()
    r = live_agent.annotate(DOC_ID)
    print()
    print({k: r[k] for k in ["read", "examples", "n_steps", "n_mapped", "n_context", "forced", "abort", "tokens_in",
                             "seconds"]})
    print("assigned:", [s["term"] for s in r["suggestions"]])
    score(r["gnd_ids"])
else:
    print("Part 3 is off. Set RUN_LIVE = True when a vLLM server is running.")

model: Qwen/Qwen3.8-27B-FP8 | prompt hash: 0d8fbfacf130

step 1 | offered ['read_abstract_intro', 'map_to_gnd']


   reasoning: Let's start by understanding the title: "Optische Spektroskopie an zweidimensionalen Lochsystemen in GaAs-AlGaAs-Quantenwells und an GaAs-AlGaAs-GaMnAs-Schichtstrukturen" Translation: "Optical spectroscopy on two-dimensional hole systems in GaAs-AlGaAs quantum wells and on GaAs-AlGaAs-GaMnAs layer structures" Main topics: - Optical spectroscopy - Two-dimensional hole systems (2D hole [...]
   model calls read_abstract_intro({})
   tool result: 9829 characters
step 2 | offered ['map_to_gnd']


   reasoning: Let me analyze the title and abstract. Title: "Optische Spektroskopie an zweidimensionalen Lochsystemen in GaAs-AlGaAs-Quantenwells und an GaAs-AlGaAs-GaMnAs-Schichtstrukturen" Main topics: 1. Optische Spektroskopie (optical spectroscopy) 2. Zweidimensionale Lochsysteme (2D hole systems) 3. GaAs-AlGaAs-Quantenwells (quantum wells) 4. GaAs-AlGaAs-GaMnAs-Schichtstrukturen (layer structures) 5. [...]
   model calls map_to_gnd({"candidates": ["Optische Spektroskopie", "Quantenwell", "Galliumarsenid", "Aluminiumgalliumarsenid", "Galliummanganarsenid", "Spintronik", "Spinrelax)


   tool result: 1530 characters
step 3 | offered ['read_toc', 'map_to_gnd', 'gnd_context']


   reasoning: Let me check the table of contents to understand the structure of the document.
   model calls read_toc({})
   tool result: 5327 characters
step 4 | offered ['map_to_gnd', 'gnd_context']


   reasoning: Let me analyze the document. The title is: "Optische Spektroskopie an zweidimensionalen Lochsystemen in GaAs-AlGaAs-Quantenwells und an GaAs-AlGaAs-GaMnAs-Schichtstrukturen" Main topics: 1. Optical spectroscopy (Optische Spektroskopie) - the method 2. Two-dimensional hole systems in GaAs-AlGaAs quantum wells 3. GaAs-AlGaAs-GaMnAs layer structures 4. Spin dynamics / spin relaxation of holes [...]
   model calls gnd_context({"gnd_ids": ["040436802", "041240103", "040191559", "1135930449", "041823338", "041489837", "042368073", "1014244218", "041233786"]})
   tool result: 3511 characters
step 5 | offered ['map_to_gnd', 'finish', 'gnd_context']


   reasoning: Let me analyze the results: 1. **Optische Spektroskopie** (040436802) - Score 1.0, assigned 2 times. This is the main method of the document. Keep. 2. **Quantenwell** (041240103) - Score 1.0, assigned 2 times. The document is about 2D hole systems in GaAs/AlGaAs quantum wells. Keep. 3. **Galliumarsenid** (040191559) - Score 1.0, assigned 6 times. Main material. Keep. 4. **AlGaAs** [...]
   model calls finish({"gnd_ids": ["040436802", "041240103", "040191559", "041823338", "041489837", "042368073"]})

{'read': ['abstract_intro', 'toc'], 'examples': ['title', 'abstract_intro', 'toc'], 'n_steps': 5, 'n_mapped': 13, 'n_context': 1, 'forced': False, 'abort': None, 'tokens_in': 25176, 'seconds': 101.33}
assigned: ['Optische Spektroskopie', 'Quantenwell', 'Galliumarsenid', 'Spinrelaxation', 'Loch (Halbleiterphysik)', 'Verdünnter magnetischer Halbleiter']
P 0.33 | R 0.67 | F1 0.44
hits:   ['Galliumarsenid', 'Optische Spektroskopie']
extra:  ['Defektelektron', 'Quantenwell', 'Semi

## Running a whole evaluation with this variant

```bash
cd ie-agent
uv run ie-agent --gpu 1 --reasoning --examples-with-sections run --name qwen_ie_examples --ids eval_ids.txt
uv run ie-agent eval --name qwen_ie_examples --compare qwen_ie_default --ids eval_ids.txt
```

The setting is stored in `settings.json` of the run, and the prompt hash differs from the default variant.